# 06 BGE-M3 blogger embedding worker

Encode one exact job artifact in the isolated 1024-dimensional BGE-M3 space.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '5275030bd9c95a523541d751605971e57ed4386d32089db3651bd068822affb7'
RUNTIME_CONTRACT = 'my-data-hub-blogger-embedding-artifact.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary runtime source for exact-revision BGE-M3 dense-only encoding."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom uuid import UUID\n\nfrom FlagEmbedding import BGEM3FlagModel\n\nfrom my_data_hub.embeddings.contracts import EmbeddingJob\nfrom my_data_hub.embeddings.models import BGE_M3\nfrom my_data_hub.embeddings.worker import EmbeddingWorker\nfrom my_data_hub.hashing import canonical_json_bytes\n\n\nclass BgeM3Encoder:\n    def __init__(self) -> None:\n        self.model = BGEM3FlagModel(BGE_M3.model_key, normalize_embeddings=True, use_fp16=False)\n\n    def encode(self, texts, *, model, max_tokens, pooling, normalize, dense_only):  # type: ignore[no-untyped-def]\n        if model != BGE_M3 or pooling != "model_native_dense" or not normalize or not dense_only:\n            raise ValueError("BGE-M3 runtime contract mismatch")\n        result = self.model.encode(list(texts), batch_size=4, max_length=max_tokens, return_dense=True, return_sparse=False, return_colbert_vecs=False)\n        return result["dense_vecs"].tolist()\n\n\ndef main() -> int:\n    payload = json.loads(Path(os.environ["MY_DATA_HUB_EMBEDDING_JOBS"]).read_text())\n    jobs = tuple(EmbeddingJob.model_validate(item) for item in payload["jobs"])\n    now = datetime.now(UTC)\n    result = EmbeddingWorker(model=BGE_M3, encoder=BgeM3Encoder()).run(\n        run_id=UUID(os.environ["MY_DATA_HUB_RUN_ID"]), jobs=jobs, started_at=now, completed_at=datetime.now(UTC)\n    )\n    Path("/kaggle/working/embedding-result.json").write_bytes(canonical_json_bytes(result.model_dump(mode="json")))\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())